# Tech Challenge - Fase 1: Análise de Expansão e Crescimento da Olist 
**(Grupo 58)**

## 1. Contexto e Pergunta de Negócio

**Pergunta Principal:** Onde estão as maiores oportunidades de expansão de receita da Olist, considerando o ticket médio e o impacto do frete?

**Justificativa Estratégica:**
No cenário do e-commerce brasileiro, o crescimento financeiro está diretamente atrelado à eficiência logística. Segundo um [levantamento publicado pela ABIACOM](https://abiacom.org/noticias/plataforma-de-intermediacao-de-fretes-e-ferramenta-importante-nas-vendas-online/), o frete é responsável por quase 66% dos custos logísticos de uma loja virtual, e o alto valor da entrega leva a 80% dos abandonos de carrinho. 

Como o modelo de negócios da Olist atua diretamente na intermediação e otimização desses contratos logísticos, através de soluções como o [Envios da Olist, que conecta pequenos e médios lojistas a transportadoras parceiras para otimizar custos](https://ajuda.olist.com/integracoes-e-automacao/envios-da-olist-%E2%80%94-guia-completo-de-integracao-e-uso), cruzar os dados de **Faturamento (Receita)** e **Frete (Custo)** me permite analisar onde estão as melhores oportunidades de expansão de mercado e onde o custo de envio está atrapalhando as vendas.

In [2]:
# Manipulação dos dados
import pandas as pd
import numpy as np

# Visualização dos dados
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

print("Ferramentas importadas com sucesso!")

Ferramentas importadas com sucesso!


In [3]:
# Lendo a base de pedidos da Olist
df_orders = pd.read_csv('../data/olist_orders_dataset.csv')

# Exibir as 3 primeiras linhas
df_orders.head(3)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00


### Entendendo a Estrutura dos Dados

Para responder à pergunta de negócio, vou relacionar as bases da Olist, começando por:

**1. Tabela de Pedidos (`df_orders`):**
Ela é o "cabeçalho" da compra. Embora não contem valores financeiros, porém contém colunas de status e data, sendo possível entender o histórico de cada caso para a análise.
* **`order_status`:** Contém status como (ex: status `delivered`) que indica quais pedidos foram entregues.
* **`order_purchase_timestamp`:** O carimbo de data. Que permite a análise ao longo do tempo (meses/anos).

In [4]:
# 1. Carregando a tabela de itens (onde estão os preços e fretes)
df_items = pd.read_csv('../data/olist_order_items_dataset.csv')

# 2. Descobrindo o tamanho da tabela (quantas linhas e colunas ela tem)
print(f"O tamanho da tabela de itens é: {df_items.shape}")

# 3. Exibindo apenas as 3 primeiras linhas para conhecermos os dados
df_items.head(3)

O tamanho da tabela de itens é: (112650, 7)


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.0,17.87


### Entendendo as Variáveis: Itens (Faturamento e Frete)
Nesta segunda tabela (`df_items`), encontramos os indicadores para responder à nossa pergunta principal de negócios:

* **`price` (Preço do Produto):** Classificada como **Variável Quantitativa Contínua**. 
  * *Justificativa Estatística:* É "Quantitativa" pois é um número real que nos permite fazer operações matemáticas (como somar o faturamento total ou calcular a média). É "Contínua" porque o valor financeiro aceita quebras e frações (casas decimais, como R$ 29,90). Representa a base do nosso Faturamento.

* **`freight_value` (Valor do Frete):** Classificada como **Variável Quantitativa Contínua**. 
  * *Justificativa Estatística:* Segue a mesma regra matemática do preço, sendo um valor financeiro que aceita frações decimais. Representa o custo logístico exato cobrado do cliente para aquela entrega.

In [6]:
# 1. Verificando se existem valores nulos (vazios) na tabela de itens (Faturamento e Frete)
print("Valores nulos na tabela de Itens:")
print(df_items.isnull().sum())

# 2. Verificando se existem valores nulos na tabela de pedidos (Datas e Status)
print("Valores nulos na tabela de Pedidos:")
print(df_orders.isnull().sum())

Valores nulos na tabela de Itens:
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64
Valores nulos na tabela de Pedidos:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64


In [9]:
# 1. Investigando os 160 pedidos sem aprovação de pagamento
print("**Status dos 160 pedidos sem Data de Aprovação **")
print(df_orders[df_orders['order_approved_at'].isnull()]['order_status'].value_counts())

print("\n**Status dos 1783 pedidos sem Envio para Transportadora **")
print(df_orders[df_orders['order_delivered_carrier_date'].isnull()]['order_status'].value_counts())

print("\n** Status dos 2965 pedidos sem Entrega ao Cliente **")
print(df_orders[df_orders['order_delivered_customer_date'].isnull()]['order_status'].value_counts())

**Status dos 160 pedidos sem Data de Aprovação **
order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64

**Status dos 1783 pedidos sem Envio para Transportadora **
order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        2
Name: count, dtype: int64

** Status dos 2965 pedidos sem Entrega ao Cliente **
order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64


### 3. Qualidade e Preparação dos Dados (Data Cleaning)

A análise de valores nulos (`.isnull().sum()`) e a investigação de status acima revelaram que:
- **Dados Financeiros Intactos:** Não há valores ausentes nas colunas `price` e `freight_value` da tabela de itens.

Além disso, é possível ver que:
- **160** pedidos estão sem data de aprovação (`order_approved_at`) e o maior número é de cancelamentos. 
- **1783** pedidos sem data de envio para a transportadora (`order_delivered_carrier_date`), indicando indisponibilidade e cancelamento com os maiores números.
- **2965** pedidos sem data de entrega ao cliente (`order_delivered_customer_date`), com destaque para os pacotes ainda em trânsito (`shipped`), cancelados e indisponíveis.

Como os dados financeiros (preço e frete) e os dados de validação (status e datas) estão separados, o meu próximo passo é unir as duas tabelas. Farei isso utilizando a coluna `order_id`, pois notei que ela é a chave identificadora comum entre ambas as bases. 

In [ ]:
# Unindo as duas tabelas usando o 'order_id' como chave de conexão
df_merged = pd.merge(df_orders, df_items, on='order_id', how='inner')

# Exibindo o tamanho da nossa nova tabela e as 3 primeiras linhas
print(f"Tamanho da nova tabela unificada: {df_merged.shape}")
df_merged.head(3)

Tamanho da nova tabela unificada: (112650, 14)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,1,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,1,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22


In [12]:
# 1. Filtrando a tabela para manter somente os pedidos com status 'delivered'
df_faturamento = df_merged[df_merged['order_status'] == 'delivered'].copy()

# 2. Mostrando o tamanho da tabela antes e depois do filtro
print(f"Tamanho antes do filtro (Todos os status): {df_merged.shape[0]} linhas")
print(f"Tamanho depois do filtro (Apenas Entregues): {df_faturamento.shape[0]} linhas")

# 3. Calculando quantos itens foram descartados (cancelados/outros)
itens_descartados = df_merged.shape[0] - df_faturamento.shape[0]
print(f"Total de itens descartados por não estarem 'entregues': {itens_descartados}")

Tamanho antes do filtro (Todos os status): 112650 linhas
Tamanho depois do filtro (Apenas Entregues): 110197 linhas
Total de itens descartados por não estarem 'entregues': 2453


### 4. Limpeza Financeira (Garantia de Receita)

Para continuar a análise, decidi verificar o que efetivamente entrou no caixa da Olist, ou seja, pedidos que foram efetivamente entregues. Por isso, filtrei a base mantendo apenas o status "delivered" (`df_merged['order_status'] == 'delivered'`).

Com esse filtro, é possível ver que:
- **2.453 itens** estavam atrelados a pedidos cancelados, não aprovados ou extraviados.
- A base final para analisar o faturamento (`df_faturamento`) conta agora com **110.197 itens** que foram vendidos e entregues de fato.

In [16]:
# 1. Calculando a soma total de Faturamento e de Frete
faturamento_total = df_faturamento['price'].sum()
frete_total = df_faturamento['freight_value'].sum()

print(f"Faturamento Total (Receita): R$ {faturamento_total:,.2f}")
print(f"Custo Total de Frete pago pelos clientes: R$ {frete_total:,.2f}")

# 2. Descobrindo o Ticket Médio e as estatísticas principais
print("\n** Resumo Estatístico por Item **")
df_faturamento[['price', 'freight_value']].describe()

Faturamento Total (Receita): R$ 13,221,498.11
Custo Total de Frete pago pelos clientes: R$ 2,198,275.64

** Resumo Estatístico por Item **


,price,freight_value
count,110197.000000,110197.000000
mean,119.980563,19.948598
std,182.299446,15.698136
min,0.850000,0.000000
25%,39.900000,13.080000
50%,74.900000,16.260000
75%,134.170000,21.150000
max,6735.000000,409.680000


In [ ]:
# 1. Criando uma nova coluna: Qual a porcentagem que o frete representa do preço do produto?
df_faturamento['taxa_frete'] = (df_faturamento['freight_value'] / df_faturamento['price']) * 100

# 2. Mostrando a média dessa nova taxa e visualizando as 3 primeiras linhas
taxa_frete_media = df_faturamento['taxa_frete'].mean()
print(f"Em média, o frete custa {taxa_frete_media:.2f}% do valor do produto.")

# 3. Mostrando como ficou a tabela com a nova coluna no final
df_faturamento[['price', 'freight_value', 'taxa_frete']].head(3)

Em média, o frete custa 32.08% do valor do produto.


,price,freight_value,taxa_frete
0,29.99,8.72,29.076359
1,118.70,22.76,19.174389
2,159.90,19.22,12.020013


In [20]:
# 1. Usando a estatística real (os quartis do .describe) para criar os "baldes"
faixas = [0, 39.90, 74.90, 134.17, df_faturamento['price'].max()]

# 2. Dando nomes reais baseados nas fatias estatísticas
nomes_faixas = [
    '1. Até R$ 39,90 (25% mais baratos)', 
    '2. R$ 39,90 a R$ 74,90', 
    '3. R$ 74,90 a R$ 134,17', 
    '4. Acima de R$ 134,17 (25% mais caros)'
]

# 3. Classificando os produtos e calculando o peso médio do frete em cada faixa
df_faturamento['categoria_preco'] = pd.cut(df_faturamento['price'], bins=faixas, labels=nomes_faixas)
prova_estatistica = df_faturamento.groupby('categoria_preco', observed=True)['taxa_frete'].mean().round(2)

print("** Estatística: Impacto do Frete baseada em Quartis **")
print(prova_estatistica)

** Estatística: Impacto do Frete baseada em Quartis **
categoria_preco
1. Até R$ 39,90 (25% mais baratos)        65.37
2. R$ 39,90 a R$ 74,90                    30.09
3. R$ 74,90 a R$ 134,17                   20.03
4. Acima de R$ 134,17 (25% mais caros)    12.34
Name: taxa_frete, dtype: float64


### 5. Análise de Incidência do Frete (Utilizando Estatística)

Para responder à minha pergunta de negócio sobre o impacto logístico, analisei a **Taxa de Incidência do Frete** (a porcentagem que o frete representa sobre o valor do produto).

A média global dessa taxa é de **32,08%**. No entanto, como notei que 75% dos fretes custam mais de R$ 13,00, decidi analisar se esse custo afeta os produtos de forma igual.

Para evitar qualquer dedução na hora de classificar o que é um produto barato ou caro, utilizei o método estatístico de **Quartis** (baseados na distribuição real extraída no início do projeto). Com isso deu para ver que:

- **Produtos Baratos (Até R$ 39,90):** O frete representa em média **65,36%** do valor da compra.
- **Produtos de Valor Médio-Baixo (R$ 39,90 a R$ 74,90):** O frete representa em média **30,09%** do valor da compra.
- **Produtos de Valor Médio-Alto (R$ 74,90 a R$ 134,17):** O frete reduz para **20,03%** do valor da compra.
- **Produtos Caros (Acima de R$ 134,17):** O frete representa apenas **12,33%** do valor da compra, tornando-se um obstáculo muito menor para a venda.

Tomando pela publicação da abiacom que fala que o alto valor da entrega é o maior causador de abandono de carrinho no e-commerce, otimizar a logística especificamente para a categoria de ticket baixo (onde o frete cobra quase o mesmo preço que o próprio produto) pode representar a maior oportunidade de expansão de receita da Olist.